# 可以训练的模型

## 数据回顾

```python
X, labels, metadata = paradigm.get_data(dataset=dataset, subjects=[1])

# X:      (576, 22, 751)  → 576个试次, 22个通道, 751个时间点
# labels: (576,)          → "left_hand", "right_hand", "foot", "tongue"
```

这是一个 **4 分类** 问题：从 EEG 信号中识别受试者在想象哪种运动。

---

## 一、传统机器学习（需先提取特征）

### 1. CSP + LDA（最经典，BCI 标杆）

```python
from sklearn.pipeline import make_pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from mne.decoding import CSP
from sklearn.model_selection import cross_val_score, KFold

pipeline = make_pipeline(
    CSP(n_components=4, reg='ledoit_wolf', log=True),  # 特征提取
    LDA()                                                # 分类器
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, labels, cv=kf)
print(f"CSP+LDA 准确率: {scores.mean():.2%}")
```

```
原理:
22通道×751时间点 → CSP提取空间特征 → 4维特征向量 → LDA分类
    原始高维           降维               低维          决策边界
```

### 2. CSP + SVM

```python
from sklearn.svm import SVC

pipeline = make_pipeline(
    CSP(n_components=6, reg='ledoit_wolf', log=True),
    SVC(kernel='rbf', C=1.0)
)
scores = cross_val_score(pipeline, X, labels, cv=kf)
print(f"CSP+SVM 准确率: {scores.mean():.2%}")
```

### 3. CSP + Random Forest / XGBoost

```python
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# CSP + 随机森林
pipeline_rf = make_pipeline(
    CSP(n_components=8, log=True),
    RandomForestClassifier(n_estimators=100)
)

# CSP + XGBoost
pipeline_xgb = make_pipeline(
    CSP(n_components=8, log=True),
    XGBClassifier(n_estimators=100, use_label_encoder=False)
)
```

---

## 二、深度学习（直接吃原始数据）

### 4. EEGNet（轻量级，专为 EEG 设计，强烈推荐）

```python
import torch
import torch.nn as nn

class EEGNet(nn.Module):
    def __init__(self, n_channels=22, n_classes=4, n_times=751):
        super().__init__()
        # 时间卷积：提取时间特征
        self.conv1 = nn.Conv2d(1, 16, (1, 64), padding=(0, 32))
        self.bn1 = nn.BatchNorm2d(16)
        
        # 深度卷积：提取空间特征（每个通道独立卷积）
        self.conv2 = nn.Conv2d(16, 32, (n_channels, 1), groups=16)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool1 = nn.AvgPool2d((1, 4))
        
        # 可分离卷积
        self.conv3 = nn.Conv2d(32, 32, (1, 16), padding=(0, 8), groups=32)
        self.conv4 = nn.Conv2d(32, 32, (1, 1))
        self.bn3 = nn.BatchNorm2d(32)
        self.pool2 = nn.AvgPool2d((1, 8))
        
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(32 * 1 * 24, n_classes)  # 尺寸需根据输入调整
    
    def forward(self, x):
        # x: (batch, 1, 22, 751)
        x = self.bn1(self.conv1(x))
        x = self.bn2(self.conv2(x))
        x = self.pool1(x)
        x = self.dropout(x)
        x = self.bn3(self.conv4(self.conv3(x)))
        x = self.pool2(x)
        x = self.dropout(x)
        x = x.flatten(1)
        return self.fc(x)
```

```
原理:
原始EEG → 时间卷积(提取频率特征) → 深度卷积(提取空间特征) → 分类
(1,22,751)    (16,22,751)          (32,1,751)           → 4类
```

### 5. ShallowConvNet（竞赛常用）

```python
class ShallowConvNet(nn.Module):
    def __init__(self, n_channels=22, n_classes=4):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 40, (1, 25))          # 时间卷积
        self.conv2 = nn.Conv2d(40, 40, (n_channels, 1)) # 空间卷积
        self.bn = nn.BatchNorm2d(40)
        self.pool = nn.AvgPool2d((1, 75), stride=(1, 15))
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(40 * 46, n_classes)          # 尺寸需调整
    
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.conv2(x)
        x = self.bn(x)
        x = torch.log(torch.clamp(x, min=1e-6))  # log 变换
        x = self.pool(x)
        x = self.dropout(x)
        return self.fc(x.flatten(1))
```

### 6. 普通 CNN

```python
class SimpleEEG_CNN(nn.Module):
    def __init__(self, n_channels=22, n_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, (3, 15), padding=(1, 7)),
            nn.ReLU(),
            nn.MaxPool2d((1, 2)),
            nn.Conv2d(32, 64, (3, 15), padding=(1, 7)),
            nn.ReLU(),
            nn.MaxPool2d((1, 2)),
            nn.Conv2d(64, 128, (3, 15), padding=(1, 7)),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(128, n_classes)
    
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x.flatten(1))
```

### 7. LSTM / GRU（捕捉时间序列依赖）

```python
class EEG_LSTM(nn.Module):
    def __init__(self, n_channels=22, n_classes=4, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(n_channels, hidden, num_layers=2, 
                           batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden * 2, n_classes)
    
    def forward(self, x):
        # x: (batch, 22, 751) → 转置为 (batch, 751, 22)
        x = x.squeeze(1).permute(0, 2, 1)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])  # 取最后时间步
```

### 8. EEGNet + LSTM 混合

```python
class EEGNet_LSTM(nn.Module):
    def __init__(self, n_channels=22, n_classes=4):
        super().__init__()
        # 先用 EEGNet 提取特征
        self.spatial = nn.Sequential(
            nn.Conv2d(1, 16, (1, 64), padding=(0, 32)),
            nn.BatchNorm2d(16),
            nn.Conv2d(16, 32, (n_channels, 1), groups=16),
            nn.BatchNorm2d(32),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
        )
        # 再用 LSTM 捕捉时间依赖
        self.lstm = nn.LSTM(32, 64, batch_first=True)
        self.fc = nn.Linear(64, n_classes)
    
    def forward(self, x):
        x = self.spatial(x)            # (batch, 32, 1, T/4)
        x = x.squeeze(2).permute(0, 2, 1)  # (batch, T/4, 32)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])
```

---

## 三、模型对比

| 模型 | 输入 | 参数量 | 准确率(4类) | 特点 |
|------|------|--------|------------|------|
| **CSP+LDA** | 需提取特征 | 极少 | ~60-65% | 经典基线，可解释性强 |
| **CSP+SVM** | 需提取特征 | 少 | ~60-68% | 比 LDA 略强 |
| **EEGNet** | 原始信号 | ~2K | ~65-72% | 轻量，BCI 专用 |
| **ShallowConvNet** | 原始信号 | ~5K | ~65-70% | 竞赛常用 |
| **CNN** | 原始信号 | ~50K | ~60-68% | 通用，需调参 |
| **LSTM** | 原始信号 | ~30K | ~58-65% | 捕捉时间依赖 |
| **EEGNet+LSTM** | 原始信号 | ~10K | ~65-73% | 空间+时间 |

> 以上准确率是 4 类运动想象的典型范围，具体取决于受试者、预处理方式和超参数。

---

## 四、快速开始（推荐先跑这个基线）

```python
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
from sklearn.pipeline import make_pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from mne.decoding import CSP
from sklearn.model_selection import cross_val_score, KFold
import numpy as np

# 加载数据
dataset = BNCI2014_001()
paradigm = MotorImagery(events=["left_hand", "right_hand", "foot", "tongue"])
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

# CSP + LDA 基线
pipeline = make_pipeline(
    CSP(n_components=4, reg='ledoit_wolf', log=True),
    LDA()
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, labels, cv=kf, scoring='accuracy')
print(f"准确率: {scores.mean():.2%} ± {scores.std():.2%}")
```

---

## 一句话总结

> **CSP+LDA 是必须先跑的基线；想要更高精度就上 EEGNet；想再进一步就用 EEGNet+LSTM 混合架构。先跑通基线，再逐步升级。**